# event-data-tools: Contact Cleaning Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aley-fiore3/event-data-tools/blob/main/demo_clean_contacts.ipynb)

This notebook walks through the core logic from `clean_contacts.py` — the main script in this repo.  
It takes a messy contact list and produces a cleaned, validated, deduplicated output.

**What this demo covers:**
- Email validation
- Name splitting and title stripping
- Duplicate detection
- Organization name normalization
- Generating a clean output CSV

No setup needed — just run the cells top to bottom.

In [ ]:
# Install dependencies (only needed in Colab)
!pip install pandas --quiet

In [ ]:
import pandas as pd
import re
from io import StringIO

print('Libraries loaded.')

## Step 1: Load a messy contact list

Here's what a typical raw contact export looks like. Common issues include:
- Full names in one column (need splitting)
- Titles mixed in ("Dr.", "Ms.", "Rev.")
- Invalid or missing emails
- Duplicate entries
- Inconsistent org names ("NYC Dept of Health", "NYC DOH", "New York City Department of Health")

In [ ]:
raw_csv = """full_name,email,organization,role
Dr. Sarah Chen,sarah.chen@greenfuture.org,Green Future Initiative,Executive Director
Marcus Williams,marcus@citycouncil.nyc.gov,NYC City Council,Policy Advisor
Rev. James Okafor,james.okafor@hopebridge.org,Hope Bridge Community Center,Program Director
Ms. Priya Patel,priya.patel@techforall.org,Tech for All,Operations Manager
sarah.chen@greenfuture.org,,Green Future Initiative,Executive Director
Tom Rivera,not-an-email,Rivera Consulting,Principal
Dr. Elena Vasquez,elena@healthpartners.org,Health Partners NYC,Medical Director
Marcus Williams,marcus@citycouncil.nyc.gov,New York City Council,Policy Advisor
Fatima Al-Hassan,fatima@youthrise.org,YouthRise,Community Organizer
Prof. David Kim,d.kim@columbia.edu,Columbia University,Associate Professor
"""

df_raw = pd.read_csv(StringIO(raw_csv))
print(f'Loaded {len(df_raw)} rows')
df_raw

## Step 2: Email validation

In [ ]:
EMAIL_PATTERN = re.compile(r'^[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}$')

def is_valid_email(email):
    if pd.isna(email) or str(email).strip() == '':
        return False
    return bool(EMAIL_PATTERN.match(str(email).strip()))

df_raw['email_valid'] = df_raw['email'].apply(is_valid_email)

print(f"Valid emails: {df_raw['email_valid'].sum()}")
print(f"Invalid/missing: {(~df_raw['email_valid']).sum()}")
df_raw[['full_name', 'email', 'email_valid']]

## Step 3: Split name and strip titles

In [ ]:
TITLES = {'dr.', 'mr.', 'mrs.', 'ms.', 'prof.', 'rev.', 'sir', 'mx.'}

def split_name(full_name):
    if pd.isna(full_name):
        return '', ''
    parts = str(full_name).strip().split()
    # Strip leading title
    if parts and parts[0].lower() in TITLES:
        parts = parts[1:]
    if len(parts) == 0:
        return '', ''
    elif len(parts) == 1:
        return parts[0], ''
    else:
        return parts[0], ' '.join(parts[1:])

df_raw[['first_name', 'last_name']] = df_raw['full_name'].apply(
    lambda n: pd.Series(split_name(n))
)

df_raw[['full_name', 'first_name', 'last_name']]

## Step 4: Detect duplicates

A row is a duplicate if it shares the same email as a row we've already seen.  
We keep the first occurrence and flag the rest.

In [ ]:
# Only deduplicate on valid emails
df_raw['dedup_flag'] = 'original'
seen_emails = set()

for idx, row in df_raw.iterrows():
    if row['email_valid']:
        email_lower = str(row['email']).lower().strip()
        if email_lower in seen_emails:
            df_raw.at[idx, 'dedup_flag'] = 'duplicate'
        else:
            seen_emails.add(email_lower)

print(f"Originals: {(df_raw['dedup_flag'] == 'original').sum()}")
print(f"Duplicates flagged: {(df_raw['dedup_flag'] == 'duplicate').sum()}")
df_raw[['full_name', 'email', 'email_valid', 'dedup_flag']]

## Step 5: Normalize organization names

The same organization often appears under multiple names. We use a mapping table —  
editable by non-technical staff — to normalize to a canonical form.

In [ ]:
ORG_ALIASES = {
    'nyc city council': 'New York City Council',
    'new york city council': 'New York City Council',
    'columbia univ': 'Columbia University',
    'columbia u': 'Columbia University',
}

def normalize_org(org):
    if pd.isna(org):
        return ''
    return ORG_ALIASES.get(str(org).strip().lower(), str(org).strip())

df_raw['organization_clean'] = df_raw['organization'].apply(normalize_org)

# Show changes
changed = df_raw[df_raw['organization'] != df_raw['organization_clean']]
print(f'{len(changed)} organization names normalized')
changed[['organization', 'organization_clean']]

## Step 6: Produce clean output

Keep only valid-email, non-duplicate rows. Output the cleaned CSV.

In [ ]:
df_clean = df_raw[
    (df_raw['email_valid'] == True) &
    (df_raw['dedup_flag'] == 'original')
].copy()

df_clean = df_clean[['first_name', 'last_name', 'email', 'organization_clean', 'role', 'email_valid', 'dedup_flag']]
df_clean = df_clean.rename(columns={'organization_clean': 'organization'})
df_clean = df_clean.reset_index(drop=True)

print(f'Input rows:  {len(df_raw)}')
print(f'Output rows: {len(df_clean)}  ({len(df_raw) - len(df_clean)} removed)')
print()
df_clean

In [ ]:
output_path = 'cleaned_contacts_output.csv'
df_clean.to_csv(output_path, index=False)
print(f'Saved to {output_path}')

# In Colab, download it:
try:
    from google.colab import files
    files.download(output_path)
    print('Download triggered.')
except ImportError:
    print(f'Not in Colab — file saved locally as {output_path}')

## Next steps

To run this on your own data:
1. Replace the `raw_csv` string with `pd.read_csv('your_file.csv')` and upload your file
2. Update `ORG_ALIASES` with your organization name variants
3. Run all cells — output downloads automatically in Colab

For the full production script (with CLI arguments, logging, and configurable options), see [`scripts/clean_contacts.py`](https://github.com/aley-fiore3/event-data-tools/blob/main/scripts/clean_contacts.py) in the main repo.

---
*[event-data-tools](https://github.com/aley-fiore3/event-data-tools) · Built by [Alessandra Desiderio](https://alessandradesiderio.com)*